<a href="https://colab.research.google.com/github/Shun0212/CodeBERTPretrained/blob/main/CompareMyCodeBERTseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers>=4.48.0
!pip install datasets

In [ ]:
from transformers import AutoModel, AutoTokenizer
import os
import sys
import torch
import random
from datasets import load_dataset, Dataset
from tokenizers import BertWordPieceTokenizer
from transformers import ModernBertConfig, ModernBertForMaskedLM, PreTrainedTokenizerFast, DataCollatorForLanguageModeling, Trainer, TrainingArguments


# モデル名
repo_name = "Shuu12121/CodeMorph-ModernBERT"
# Hugging Face からモデルをロード
model = ModernBertForMaskedLM.from_pretrained(repo_name)
tokenizer = AutoTokenizer.from_pretrained(repo_name)

print("モデルのロード成功！")
print(model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
model.to(device)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/609M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/408k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
     

ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, bias=False)
        )
      )
      (1-11): 11

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model=model, tokenizer=tokenizer)
print(fill_mask("def add_numbers(a, b): return a + [MASK]"))

Device set to use cuda:0


[{'score': 0.9981516003608704, 'token': 49908, 'token_str': 'b', 'sequence': 'def add _ numbers ( a, b ) : return a + b'}, {'score': 0.0010269464692100883, 'token': 49886, 'token_str': 'a', 'sequence': 'def add _ numbers ( a, b ) : return a + a'}, {'score': 0.00034104351652786136, 'token': 49891, 'token_str': 'c', 'sequence': 'def add _ numbers ( a, b ) : return a + c'}, {'score': 8.821619121590629e-05, 'token': 49938, 'token_str': '1', 'sequence': 'def add _ numbers ( a, b ) : return a + 1'}, {'score': 2.8221440516063012e-05, 'token': 159, 'token_str': 'end', 'sequence': 'def add _ numbers ( a, b ) : return a + end'}]


In [ ]:
import torch

def get_embedding(text, model, tokenizer, device="cuda"):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256)
    # token_type_ids があれば削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.model(**inputs)
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding

embedding = get_embedding("def my_function(): pass", model, tokenizer)
print(embedding.shape)


torch.Size([1, 768])


In [ ]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデル・各言語でコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    # Python の乱数シードを設定（候補サンプルの抽出などに影響します）
    random.seed(35)
    np.random.seed(35)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットの言語一覧（例）
    # ------------------------------
    languages = ["python", "java", "javascript", "php", "ruby", "go"]
    max_examples = 1000  # 各言語ごとに先頭100サンプルを利用（ランダムサンプリング後）

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERTv2", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-large-vocab", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-Alternative", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
    ]

    for lang in languages:
        print(f"\n==== 言語: {lang} の評価を開始 ====")
        # データセットをロード後、シャッフルしてランダムサンプルを抽出
        dataset = load_dataset("google/code_x_glue_ct_code_to_text", lang, split="test", trust_remote_code=True)
        dataset = dataset.shuffle(seed=35)
        subset = dataset.select(range(max_examples))

        for config in model_configs:
            model_name = config["name"]
            model_class = config["class"]
            print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = model_class.from_pretrained(model_name)
            model.to(device)

            metrics = evaluate_code_search(model, tokenizer, subset, device,
                                           max_examples=max_examples,
                                           pool_size=100,
                                           query_field="docstring",
                                           code_field="code")
            display_code_search_results(metrics, f"{model_name} - {lang}")


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

README.md:   0%|          | 0.00/26.7k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/147M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/7325 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8122 [00:00<?, ? examples/s]


Shuu12121/CodeMorph-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT - go Code Search Evaluation Results ====
MRR:         0.5358
MAP:         0.5358
R-Precision: 0.4320

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4320    0.4320    0.4320    0.4320    0.4320         0.4320         
5     0.6490    0.5131    0.5470    0.5477    0.6490         0.6490         
10    0.7320    0.5243    0.5739    0.5674    0.7320         0.7320         
50    0.9520    0.5351    0.6233    0.5879    0.9520         0.9520         
100   1.0000    0.5358    0.6311    0.5893    1.0000         1.0000         

Shuu12121/CodeMorph-ModernBERTv2 を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERTv2 - go Code Search Evaluation

You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal - go Code Search Evaluation Results ====
MRR:         0.8227
MAP:         0.8227
R-Precision: 0.7470

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.7470    0.7470    0.7470    0.7470    0.7470         0.7470         
5     0.9240    0.8154    0.8426    0.8438    0.9240         0.9240         
10    0.9630    0.8207    0.8553    0.8531    0.9630         0.9630         
50    0.9970    0.8226    0.8633    0.8567    0.9970         0.9970         
100   1.0000    0.8227    0.8638    0.8568    1.0000         1.0000         


In [ ]:
import torch
import numpy as np
import random
import re
from datasets import load_dataset
from transformers import AutoModelForMaskedLM, AutoTokenizer, T5ForConditionalGeneration
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# ユーティリティ関数群
# ---------------------------------------------------

def remove_multiline_strings(code):
    """
    Pythonコード中のマルチライン文字列（""" """ や ''' '''）を削除します。
    """
    pattern = r'("""(.*?)"""|\'\'\'(.*?)\'\'\')'
    return re.sub(pattern, '', code, flags=re.DOTALL)

def get_cls_embedding(model, tokenizer, text, device, max_length=256):
    """
    入力テキストの埋め込みを取得する関数。
    - BERT/CodeBERT系の場合は CLS トークンの埋め込みを使用。
    - CodeT5系の場合は encoder 出力の平均プーリングを使用。
    - ModernBERT のように内部に model がある場合は model.model を利用。
    """
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    # ModernBERT の場合、token_type_ids が不要なので削除する
    if "token_type_ids" in inputs:
        inputs.pop("token_type_ids")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    if hasattr(model, "model"):
        outputs = model.model(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "bert"):
        outputs = model.bert(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "roberta"):
        outputs = model.roberta(**inputs)
        embedding = outputs.last_hidden_state[:, 0, :]
    elif hasattr(model, "encoder"):
        # CodeT5系の場合は、CLSトークンが存在しないため平均プーリングを使用
        outputs = model.encoder(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1)
    else:
        outputs = model(**inputs)
        if hasattr(outputs, "last_hidden_state"):
            embedding = outputs.last_hidden_state[:, 0, :]
        elif hasattr(outputs, "hidden_states"):
            embedding = outputs.hidden_states[-1][:, 0, :]
        else:
            raise ValueError("モデル出力にlast_hidden_stateまたはhidden_statesがありません。")

    return embedding.detach().cpu().numpy()


def calculate_f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def calculate_dcg(ranking):
    dcg = 0.0
    for i, rel in enumerate(ranking):
        dcg += rel / np.log2(i + 2)
    return dcg

def calculate_metrics(sim_matrix, k_values=[1, 5, 10, 50, 100]):
    """
    候補プール内で正解（常にインデックス0が正解と仮定）に基づいて評価指標を計算します。
    """
    num_queries = sim_matrix.shape[0]
    metrics = {
        "mrr": 0.0,
        "map": 0.0,
        "r_precision": 0.0,
        "recall@k": {k: 0.0 for k in k_values},
        "precision@k": {k: 0.0 for k in k_values},
        "ndcg@k": {k: 0.0 for k in k_values},
        "f1@k": {k: 0.0 for k in k_values},
        "success_rate@k": {k: 0.0 for k in k_values},
        "query_coverage@k": {k: 0.0 for k in k_values},
    }

    for i in range(num_queries):
        sims = sim_matrix[i]
        ranked_indices = np.argsort(-sims)
        # 正解は候補プール内の先頭（インデックス0）
        correct_rank = np.where(ranked_indices == 0)[0][0] + 1
        metrics["mrr"] += 1.0 / correct_rank
        metrics["map"] += 1.0 / correct_rank
        metrics["r_precision"] += 1.0 if correct_rank == 1 else 0.0

        for k in k_values:
            if correct_rank <= k:
                prec = 1.0 / correct_rank
                rec = 1.0
            else:
                prec = 0.0
                rec = 0.0
            f1 = calculate_f1(prec, rec)
            metrics["precision@k"][k] += prec
            metrics["recall@k"][k] += rec
            metrics["f1@k"][k] += f1

            ideal_ranking = [1.0] + [0.0] * (k - 1)
            actual_ranking = [1.0 if j == 0 else 0.0 for j in ranked_indices[:k]]
            idcg = calculate_dcg(ideal_ranking)
            dcg = calculate_dcg(actual_ranking)
            metrics["ndcg@k"][k] += dcg / idcg if idcg > 0 else 0.0

            metrics["success_rate@k"][k] += 1.0 if correct_rank <= k else 0.0
            metrics["query_coverage@k"][k] += 1.0 if correct_rank <= k else 0.0

    metrics["mrr"] /= num_queries
    metrics["map"] /= num_queries
    metrics["r_precision"] /= num_queries
    for k in k_values:
        metrics["precision@k"][k] /= num_queries
        metrics["recall@k"][k] /= num_queries
        metrics["ndcg@k"][k] /= num_queries
        metrics["f1@k"][k] /= num_queries
        metrics["success_rate@k"][k] /= num_queries
        metrics["query_coverage@k"][k] /= num_queries
    return metrics

def display_code_search_results(metrics, model_name):
    """
    評価結果を見やすいテーブル形式で表示します。
    """
    print(f"\n==== {model_name} Code Search Evaluation Results ====")
    print(f"MRR:         {metrics['mrr']:.4f}")
    print(f"MAP:         {metrics['map']:.4f}")
    print(f"R-Precision: {metrics['r_precision']:.4f}\n")

    header = "{:<6}{:<10}{:<10}{:<10}{:<10}{:<15}{:<15}"
    print(header.format("K", "Recall", "Precision", "NDCG", "F1", "Success Rate", "Query Cov."))
    print("-" * 76)
    for k in sorted(metrics["recall@k"].keys()):
        recall = metrics["recall@k"][k]
        precision = metrics["precision@k"][k]
        ndcg = metrics["ndcg@k"][k]
        f1 = metrics["f1@k"][k]
        success_rate = metrics["success_rate@k"][k]
        query_cov = metrics["query_coverage@k"][k]
        print(header.format(k,
                            f"{recall:.4f}",
                            f"{precision:.4f}",
                            f"{ndcg:.4f}",
                            f"{f1:.4f}",
                            f"{success_rate:.4f}",
                            f"{query_cov:.4f}"))

def evaluate_code_search(model, tokenizer, dataset, device, max_examples=100, pool_size=100,
                         query_field="docstring", code_field="code"):
    """
    指定したデータセット上でコード検索タスクを実施し、評価指標を計算します。
    各サンプルに対して、正解は候補プール内の先頭（自身）と仮定しています。
    """
    num_examples = min(len(dataset), max_examples)
    all_codes = [remove_multiline_strings(dataset[i][code_field]) for i in range(num_examples)]
    queries = [dataset[i][query_field] for i in range(num_examples)]

    # 各コードの埋め込み計算
    all_code_embeddings = []
    print("データセット全体のコード埋め込みを計算中...")
    for code in all_codes:
        emb = get_cls_embedding(model, tokenizer, code, device)
        all_code_embeddings.append(emb)
    all_code_embeddings = np.concatenate(all_code_embeddings, axis=0)
    print("コード埋め込み計算完了.")

    # 各クエリに対し候補プール内の類似度計算
    sim_matrices = []
    print("候補コードプールを作成し、類似度計算中...")
    for i in range(num_examples):
        query_embedding = get_cls_embedding(model, tokenizer, queries[i], device).reshape(1, -1)
        candidate_indices = [i] + random.sample([j for j in range(num_examples) if j != i], pool_size - 1)
        candidate_embeddings = all_code_embeddings[candidate_indices]
        sims = cosine_similarity(query_embedding, candidate_embeddings)[0]
        sim_matrices.append(sims)
    sim_matrix = np.array(sim_matrices)
    print("類似度計算完了.")

    metrics = calculate_metrics(sim_matrix)
    return metrics

# ---------------------------------------------------
# メイン処理：各モデルでコード検索実験を実施
# ---------------------------------------------------

if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # ------------------------------
    # ① ModernBERT 単体ロード確認
    # ------------------------------
    repo_name = "Shuu12121/CodeMorph-ModernBERT"
    model_demo = AutoModelForMaskedLM.from_pretrained(repo_name)
    tokenizer_demo = AutoTokenizer.from_pretrained(repo_name)
    model_demo.to(device)
    print("\n【ModernBERT 単体ロード確認】")
    print("モデルのロード成功！")
    print(model_demo)

    # ------------------------------
    # ② 評価データセットのロード
    # ------------------------------
    print("\ngoogle/code_x_glue_tc_nl_code_search_adv データセット (Test) をロードします...")
    tc_dataset = load_dataset("google/code_x_glue_tc_nl_code_search_adv", split="test", trust_remote_code=True)
    dataset = tc_dataset.shuffle(seed=35)
    max_examples= 19210
    subset = dataset.select(range(max_examples))

    # ------------------------------
    # ③ 各モデルでコード検索実験を実施
    # ------------------------------
    model_configs = [
        {"name": "microsoft/graphcodebert-base", "class": AutoModelForMaskedLM},
        {"name": "microsoft/codebert-base-mlm", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERTv2", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-large-vocab", "class": AutoModelForMaskedLM},
        {"name": "Shuu12121/CodeMorph-ModernBERT-Alternative", "class": AutoModelForMaskedLM},
        {"name": "Salesforce/codet5p-220m-bimodal", "class": T5ForConditionalGeneration},
    ]

    for config in model_configs:
        model_name = config["name"]
        model_class = config["class"]
        print(f"\n{model_name} を評価します (候補プールサイズ: 100)...")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = model_class.from_pretrained(model_name)
        model.to(device)

        metrics = evaluate_code_search(model, tokenizer, subset, device,
                                       max_examples=19210,
                                       pool_size=100,
                                       query_field="docstring",
                                       code_field="code")
        display_code_search_results(metrics, model_name)


使用デバイス: cuda

【ModernBERT 単体ロード確認】
モデルのロード成功！
ModernBertForMaskedLM(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50000, 768, padding_idx=0)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (rotary_emb): ModernBertRotaryEmbedding()
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=3072, out_features=768, 

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

train-00000-of-00002.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

train-00001-of-00002.parquet:   0%|          | 0.00/147M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/8.59M [00:00<?, ?B/s]

...

Some weights of the model checkpoint at microsoft/codebert-base-mlm were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== microsoft/codebert-base-mlm Code Search Evaluation Results ====
MRR:         0.3559
MAP:         0.3559
R-Precision: 0.2528

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.2528    0.2528    0.2528    0.2528    0.2528         0.2528         
5     0.4466    0.3234    0.3540    0.3541    0.4466         0.4466         
10    0.5593    0.3382    0.3903    0.3802    0.5593         0.5593         
50    0.8987    0.3545    0.4655    0.4109    0.8987         0.8987         
100   1.0000    0.3559    0.4821    0.4139    1.0000         1.0000         

Shuu12121/CodeMorph-ModernBERT を評価します (候補プールサイズ: 100)...
データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Shuu12121/CodeMorph-ModernBERT Code Search Evaluation Results ====
MRR:         0.6196
MAP:         0.6196
R-Precision: 0.5136



You are using a model of type codet5p_bimodal to instantiate a model of type t5. This is not supported for all configurations of models and can yield errors.


データセット全体のコード埋め込みを計算中...
コード埋め込み計算完了.
候補コードプールを作成し、類似度計算中...
類似度計算完了.

==== Salesforce/codet5p-220m-bimodal Code Search Evaluation Results ====
MRR:         0.5336
MAP:         0.5336
R-Precision: 0.4221

K     Recall    Precision NDCG      F1        Success Rate   Query Cov.     
----------------------------------------------------------------------------
1     0.4221    0.4221    0.4221    0.4221    0.4221         0.4221         
5     0.6590    0.5106    0.5476    0.5484    0.6590         0.6590         
10    0.7518    0.5232    0.5778    0.5704    0.7518         0.7518         
50    0.9463    0.5328    0.6214    0.5886    0.9463         0.9463         
100   1.0000    0.5336    0.6302    0.5902    1.0000         1.0000         


In [ ]:
# prompt: 切断する

from google.colab import runtime
runtime.unassign()
